# Qwen2-Audio baseline and int8 quantization


In [ ]:
# ============================================================
# CELL 1: Imports and global config
# ============================================================

from pathlib import Path
import gc
import time

import pandas as pd
import soundfile as sf
import torch
from tqdm.auto import tqdm

from transformers import (
    AutoProcessor,
    Qwen2AudioForConditionalGeneration,
    BitsAndBytesConfig,
)

from comet import download_model, load_from_checkpoint


PROJECT_ROOT = Path("/home/paperspace/projects/iwslt2026-compression")
OUT_DIR = PROJECT_ROOT / "outputs" / "qwen2audio_full_en_zh"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ID = "Qwen/Qwen2-Audio-7B-Instruct"
DEVICE = "cuda:0"
PROMPT_TEXT = "Translate this English speech into Chinese."
MAX_NEW_TOKENS = 64
COMET_BATCH_SIZE = 16

DEV_MANIFEST = PROJECT_ROOT / "data" / "manifests" / "acl6060_dev_en_zh.csv"
EVAL_MANIFEST = PROJECT_ROOT / "data" / "manifests" / "acl6060_eval_en_zh.csv"

print("project root exists:", PROJECT_ROOT.exists())
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    
    
# ============================================================
# CELL 3: Helper functions only
# ============================================================

def generate_translation(model, processor, audio_path, prompt_text=PROMPT_TEXT, max_new_tokens=MAX_NEW_TOKENS):
    audio, sr = sf.read(str(audio_path))

    if audio.ndim > 1:
        audio = audio.mean(axis=1)

    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "audio", "audio_url": str(audio_path)},
                {"type": "text", "text": prompt_text},
            ],
        }
    ]

    text = processor.apply_chat_template(
        conversation,
        add_generation_prompt=True,
        tokenize=False,
    )

    inputs = processor(
        text=text,
        audios=[audio],
        sampling_rate=sr,
        return_tensors="pt",
    )

    inputs = {
        k: v.to(DEVICE) if hasattr(v, "to") else v
        for k, v in inputs.items()
    }

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            use_cache=True,
        )

    generated_ids = generated_ids[:, inputs["input_ids"].size(1):]

    pred = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0].strip()

    return pred


def run_inference(model, df_input, run_name):
    preds = []
    start = time.time()

    for _, row in tqdm(df_input.iterrows(), total=len(df_input), desc=run_name):
        pred = generate_translation(model, processor, Path(row["audio_path"]))
        preds.append(pred)

    elapsed = time.time() - start

    out_df = df_input.copy()
    out_df["prediction"] = preds
    return out_df, elapsed


def compute_comet(pred_df, comet_model, batch_size=COMET_BATCH_SIZE):
    comet_data = [
        {"src": r["source_en"], "mt": r["prediction"], "ref": r["target_zh"]}
        for _, r in pred_df.iterrows()
    ]

    comet_out = comet_model.predict(comet_data, batch_size=batch_size, gpus=1)
    comet_score = comet_out.system_score if hasattr(comet_out, "system_score") else comet_out[1]
    return comet_score


def cleanup_model(model):
    del model
    gc.collect()
    torch.cuda.empty_cache()


# ============================================================
# CELL 2: Data and shared objects
# ============================================================

df = pd.concat(
    [
        pd.read_csv(DEV_MANIFEST),
        pd.read_csv(EVAL_MANIFEST),
    ],
    ignore_index=True,
)

df["audio_path"] = df["audio_path"].apply(lambda p: str((PROJECT_ROOT / p).resolve()))

processor = AutoProcessor.from_pretrained(MODEL_ID)

comet_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(comet_path)

print("rows:", len(df))
print(df.head(2))



# ============================================================
# CELL 4: Baseline run + COMET + cleanup
# ============================================================

baseline_model = Qwen2AudioForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
baseline_model.eval()

baseline_df, baseline_time = run_inference(baseline_model, df, "baseline_fp16")

baseline_out = OUT_DIR / "baseline_full_preds.csv"
baseline_df.to_csv(baseline_out, index=False, encoding="utf-8")

baseline_comet = compute_comet(baseline_df, comet_model)

print("baseline predictions saved to:", baseline_out)
print("baseline seconds:", round(baseline_time, 2))
print("baseline COMET:", baseline_comet)

cleanup_model(baseline_model)


# ============================================================
# CELL 5: Int8 run + COMET + cleanup
# ============================================================

bnb_config = BitsAndBytesConfig(load_in_8bit=True)

quant_model = Qwen2AudioForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
quant_model.eval()

quant_df, quant_time = run_inference(quant_model, df, "quantized_int8")

quant_out = OUT_DIR / "quant8_full_preds.csv"
quant_df.to_csv(quant_out, index=False, encoding="utf-8")

quant_comet = compute_comet(quant_df, comet_model)

print("int8 predictions saved to:", quant_out)
print("int8 seconds:", round(quant_time, 2))
print("int8 COMET:", quant_comet)

cleanup_model(quant_model)


# ============================================================
# CELL 6: Final summary
# ============================================================

summary = pd.DataFrame(
    [
        {
            "run": "baseline_fp16",
            "rows": len(baseline_df),
            "seconds": baseline_time,
            "comet": baseline_comet,
        },
        {
            "run": "quantized_int8",
            "rows": len(quant_df),
            "seconds": quant_time,
            "comet": quant_comet,
        },
    ]
)

baseline_score = summary.loc[summary["run"] == "baseline_fp16", "comet"].iloc[0]
summary["comet_diff_vs_baseline"] = summary["comet"] - baseline_score
summary["speedup_vs_baseline"] = baseline_time / summary["seconds"]

summary_out = OUT_DIR / "summary_comet_only.csv"
summary.to_csv(summary_out, index=False)

print("summary saved to:", summary_out)
summary